# Cat and Dog Image Classification using MobileNetV2 and ONNX Export

This notebook trains a binary image classifier (Cats vs. Dogs) using transfer learning with MobileNetV2 in TensorFlow, evaluates it on a test set, and exports the final trained model into the **ONNX** format instead of `.keras`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.utils import img_to_array, load_img
import matplotlib.pyplot as plt
import numpy as np
import os
from PIL import Image
import tf2onnx
import onnx

print("TensorFlow version:", tf.__version__)

## 1. Load Dataset

In [ ]:
possible_paths = [
    './dataset',
    '../dataset',
    os.path.join(os.getcwd(), 'dataset')
]

data_dir = None
for path in possible_paths:
    if os.path.exists(path):
        data_dir = path
        break

if data_dir:
    print(f"✅ Dataset found at: {os.path.abspath(data_dir)}")
else:
    print("❌ Dataset not found! Make sure your 'dataset' folder contains 'cats' and 'dogs' subfolders.")

BATCH_SIZE = 32
IMG_SIZE = (160, 160)
MAX_IMAGES_PER_CLASS = 1000  # Adjust lower if you hit memory issues

In [ ]:
print(f"Loading up to {MAX_IMAGES_PER_CLASS} images per class...")

images = []
labels = []
class_names = sorted(os.listdir(data_dir))
class_names = [c for c in class_names if os.path.isdir(os.path.join(data_dir, c))]

for idx, class_name in enumerate(class_names):
    class_path = os.path.join(data_dir, class_name)
    files = os.listdir(class_path)
    
    np.random.seed(123)
    np.random.shuffle(files)
    
    count = 0
    for file in files:
        if count >= MAX_IMAGES_PER_CLASS:
            break
            
        file_path = os.path.join(class_path, file)
        try:
            with Image.open(file_path) as img:
                img = img.convert('RGB')
                img = img.resize(IMG_SIZE)
                arr = img_to_array(img)
                images.append(arr)
                labels.append(idx)
                count += 1
        except:
            pass

images = np.array(images, dtype=np.float32)
labels = np.array(labels, dtype=np.int32)

print(f"✅ Total images loaded: {len(images)} | Classes: {class_names}")

## 2. Split Dataset and Create tf.data.Dataset

In [ ]:
# Shuffle indices
indices = np.arange(len(images))
np.random.seed(123)
np.random.shuffle(indices)

images = images[indices]
labels = labels[indices]

# Split Ratios: 70% Train, 15% Validation, 15% Test
train_split = int(0.70 * len(images))
val_split = int(0.85 * len(images))

train_images, val_images, test_images = images[:train_split], images[train_split:val_split], images[val_split:]
train_labels, val_labels, test_labels = labels[:train_split], labels[train_split:val_split], labels[val_split:]

print(f"Dataset Splits -> Train: {len(train_images)} | Val: {len(val_images)} | Test: {len(test_images)}")

# Create tf.data.Dataset objects
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_labels)).shuffle(1000).batch(BATCH_SIZE)
validation_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_labels)).batch(BATCH_SIZE)

AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.prefetch(buffer_size=AUTOTUNE)

## 3. Build Model using MobileNetV2 Transfer Learning

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)

# Freeze base model weights
base_model.trainable = False

# Construct classifier head
model = models.Sequential([
    layers.Rescaling(1./127.5, offset=-1, input_shape=IMG_SIZE + (3,)),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid') # Binary classification (Cat vs Dog)
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Train the Model

In [ ]:
EPOCHS = 10

history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

## 5. Export Model to ONNX Format

In [ ]:
onnx_path = 'cat_dog_classifier.onnx'

# Specify the input signature with dynamic batch size
input_signature = [tf.TensorSpec((None, IMG_SIZE[0], IMG_SIZE[1], 3), tf.float32, name='image_input')]

# Convert the Keras model to ONNX format
model_proto, _ = tf2onnx.convert.from_keras(model, input_signature=input_signature, opset=13)

# Save the ONNX model to disk
onnx.save(model_proto, onnx_path)
print(f"Model saved successfully as '{onnx_path}'!")

## 6. Evaluate and Visualize Predictions

In [ ]:
# 1. Evaluate accuracy on the unused test set
test_dataset = tf.data.Dataset.from_tensor_slices((test_images, test_labels)).batch(BATCH_SIZE)
test_loss, test_acc = model.evaluate(test_dataset)
print(f"\n🎯 Final Test Set Accuracy: {test_acc * 100:.2f}%")

# 2. Visualize predictions on random test images
num_images_to_show = 6
plt.figure(figsize=(12, 8))

random_indices = np.random.choice(len(test_images), min(num_images_to_show, len(test_images)), replace=False)

for i, idx in enumerate(random_indices):
    img_array = test_images[idx]
    true_label_idx = test_labels[idx]
    true_label_name = class_names[true_label_idx]
    
    # Predict
    prediction_input = tf.expand_dims(img_array, 0)
    score = model.predict(prediction_input, verbose=0)[0][0]
    
    if score > 0.5:
        pred_label_name = class_names[1] # Dog
        confidence = score * 100
    else:
        pred_label_name = class_names[0] # Cat
        confidence = (1 - score) * 100
        
    # Plot
    plt.subplot(2, 3, i + 1)
    plt.imshow(img_array.astype('uint8'))
    plt.axis('off')
    
    color = 'green' if pred_label_name == true_label_name else 'red'
    plt.title(f"True: {true_label_name}\nPred: {pred_label_name} ({confidence:.1f}%)", color=color)

plt.tight_layout()
plt.show()